# Voice bake-off: English debate with correctly pronounced Sanskrit

Renders the same test lines several ways so they can be compared by ear before anything is wired into the player.

| Variant | Engine | Input |
|---|---|---|
| **A** | Indic Parler-TTS, English voice | English with Sanskrit inline in Kannada script |
| **B** | Indic Parler-TTS, Kannada voice | English with Sanskrit inline in Kannada script |
| **C** | Indic Parler-TTS, English voice | English and Sanskrit spans rendered separately and joined |
| **D** | Kokoro (current engine) | Sanskrit spans given as rule-based IPA |
| **Q** | Indic Parler-TTS | Scripture quote in Kannada script: Sanskrit voice, then the debater's voice |

Sanskrit is routed through Kannada script (the Vāgdhenu trick) so the model keeps the inherent vowel instead of applying Hindi schwa deletion. Sanskrit-only clips (C spans and Q) are transcribed by IndicConformer and the best of `N_TAKES` takes is kept.

**Before running**
1. *Runtime → Change runtime type → T4 GPU.*
2. On Hugging Face, accept the terms for [indic-parler-tts](https://huggingface.co/ai4bharat/indic-parler-tts) and [indic-conformer-600m-multilingual](https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual), then create a read token.
3. Add the token as a Colab secret named `HF_TOKEN` (key icon in the left bar), or paste it when prompted.

If an import fails after the install cell, use *Runtime → Restart session* and run from the top again. Files in `/content` survive a restart.

## 1. Setup

In [ ]:
!pip install -q git+https://github.com/huggingface/parler-tts.git
!pip install -q kokoro "misaki[en]" soundfile librosa
!pip install -q onnxruntime onnx  # IndicConformer; CPU build (the 1.20.1 pins have no Python 3.13 wheels)
!pip install -q -U "protobuf>=5.26.1"  # the installs above can downgrade it; transformers needs runtime_version
!apt-get -qq install -y espeak-ng > /dev/null

In [ ]:
import torch
from huggingface_hub import login

try:
    from google.colab import userdata
    login(userdata.get('HF_TOKEN'))
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('device:', device, '(rendering will be very slow on CPU)' if device == 'cpu' else '')

## 2. Script helpers
Devanagari ↔ Kannada routing, Vāgdhenu-style Sanskrit rules (homorganic anusvāra, final visarga echo), Kannada → IPA for Kokoro, and span splitting.

In [ ]:
# Script helpers: Devanagari <-> Kannada routing, Vagdhenu-style Sanskrit rules,
# Kannada -> IPA for Kokoro, and span splitting for mixed English/Sanskrit text.
import re

VIRAMA, ANUSVARA, VISARGA = '್', 'ಂ', 'ಃ'
KN_SPAN = re.compile(r'[ಀ-೿]+(?:\s+[ಀ-೿]+)*')
KN_CONS = 'ಕ-ಹ'

# Devanagari and Kannada share the ISCII layout, so the mapping is a fixed offset.
DEVA_SPECIAL = {'।': '.', '॥': '.', 'ॐ': 'ಓಂ', '़': '', 'ँ': 'ಂ'}

def deva_to_kn(text):
    out = []
    for ch in text:
        if ch in DEVA_SPECIAL:
            out.append(DEVA_SPECIAL[ch])
        elif 0x0900 <= ord(ch) <= 0x097F:
            out.append(chr(ord(ch) + 0x380))
        else:
            out.append(ch)
    return ''.join(out)

def kn_to_deva(text):
    return ''.join(chr(ord(c) - 0x380) if 0x0C80 <= ord(c) <= 0x0CFF else c for c in text)

# Anusvara before a stop becomes the nasal of that stop's class (parasavarna).
NASAL_OF = {}
for start, nasal in [(0x0C95, 'ಙ'), (0x0C9A, 'ಞ'), (0x0C9F, 'ಣ'), (0x0CA4, 'ನ'), (0x0CAA, 'ಮ')]:
    for o in range(start, start + 5):
        NASAL_OF[chr(o)] = nasal

def sanskrit_rules(kn):
    kn = re.sub(ANUSVARA + r'(\s*)([' + ''.join(NASAL_OF) + '])',
                lambda m: NASAL_OF[m[2]] + VIRAMA + m[1] + m[2], kn)
    # Final visarga after a short vowel is echoed (hetuh -> hetuhu); after a long
    # vowel it is left bare.
    end = r'(?=[\s.,;:!?]|$)'
    kn = re.sub(r'(?<=[' + KN_CONS + '])' + VISARGA + end, 'ಹ', kn)
    kn = re.sub('ಿ' + VISARGA + end, 'ಿಹಿ', kn)
    kn = re.sub('ು' + VISARGA + end, 'ುಹು', kn)
    return kn

def split_spans(text):
    """[(is_sanskrit, piece), ...] with punctuation-only pieces folded into neighbours."""
    pieces, last = [], 0
    for m in KN_SPAN.finditer(text):
        if m.start() > last:
            pieces.append((False, text[last:m.start()]))
        pieces.append((True, m.group()))
        last = m.end()
    if last < len(text):
        pieces.append((False, text[last:]))
    out = []
    for sa, p in pieces:
        if not sa and out:
            lead = re.match(r'\s*[^\w\s]*', p).group().strip()
            if lead:
                out[-1] = (out[-1][0], out[-1][1] + lead)
                p = p.lstrip()[len(lead):]
        if p.strip():
            out.append((sa, p.strip()))
    return out

# Kannada script -> IPA, with Kokoro's English-oriented symbol choices.
IPA_VOWEL = {'ಅ': 'ʌ', 'ಆ': 'ɑː', 'ಇ': 'ɪ', 'ಈ': 'iː', 'ಉ': 'ʊ', 'ಊ': 'uː',
             'ಋ': 'ɹɪ', 'ೠ': 'ɹiː', 'ಎ': 'e', 'ಏ': 'eː', 'ಐ': 'aɪ',
             'ಒ': 'o', 'ಓ': 'oː', 'ಔ': 'aʊ'}
IPA_SIGN = {'ಾ': 'ɑː', 'ಿ': 'ɪ', 'ೀ': 'iː', 'ು': 'ʊ', 'ೂ': 'uː', 'ೃ': 'ɹɪ',
            'ೄ': 'ɹiː', 'ೆ': 'e', 'ೇ': 'eː', 'ೈ': 'aɪ', 'ೊ': 'o', 'ೋ': 'oː',
            'ೌ': 'aʊ'}
IPA_CONS = dict(zip(
    'ಕಖಗಘಙಚಛಜಝಞಟಠಡಢಣತಥದಧನಪಫಬಭಮಯರಲವಶಷಸಹಳ',
    ['k', 'kʰ', 'ɡ', 'ɡʰ', 'ŋ', 'ʧ', 'ʧʰ', 'ʤ', 'ʤʰ', 'ɲ', 'ʈ', 'ʈʰ', 'ɖ', 'ɖʰ', 'ɳ',
     't', 'tʰ', 'd', 'dʰ', 'n', 'p', 'pʰ', 'b', 'bʰ', 'm', 'j', 'ɹ', 'l', 'ʋ', 'ʃ', 'ʂ', 's', 'h', 'ɭ']))
# Used when a symbol is missing from the Kokoro vocabulary.
IPA_FALLBACK = {'ʰ': '', 'ʈ': 't', 'ɖ': 'd', 'ɳ': 'n', 'ʂ': 'ʃ', 'ɲ': 'n', 'ʋ': 'v', 'ɭ': 'l', 'ʌ': 'ə', 'ɡ': 'g'}

def kn_to_ipa(word):
    out, i = [], 0
    while i < len(word):
        ch = word[i]
        nxt = word[i + 1] if i + 1 < len(word) else ''
        if ch in IPA_CONS:
            out.append(IPA_CONS[ch])
            if nxt == VIRAMA:
                i += 1
            elif nxt in IPA_SIGN:
                out.append(IPA_SIGN[nxt])
                i += 1
            else:
                out.append('ə' if i + 1 == len(word) else 'ʌ')
        elif ch in IPA_VOWEL:
            out.append(IPA_VOWEL[ch])
        elif ch == ANUSVARA:
            out.append('m' if i + 1 == len(word) else 'ŋ')
        elif ch == VISARGA:
            out.append('h')
        i += 1
    return 'ˈ' + ''.join(out)

def fit_vocab(ipa, vocab):
    if vocab is None:
        return ipa
    out = []
    for ch in ipa:
        if ch in vocab or ch == 'ˈ':
            out.append(ch)
        else:
            out.append(IPA_FALLBACK.get(ch, ''))
    return ''.join(out)

def kokoro_text(text, vocab=None, rules=True):
    """English with Sanskrit spans replaced by Kokoro's [word](/phonemes/) markup."""
    def link(m):
        span = sanskrit_rules(m.group()) if rules else m.group()
        ipa = ' '.join(fit_vocab(kn_to_ipa(w), vocab) for w in span.split())
        return f'[{m.group()}](/{ipa}/)'
    return KN_SPAN.sub(link, text)

def levenshtein(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def norm_deva(s):
    s = re.sub(r'[^ऀ-ॣ०-ॿ]', '', s)
    return re.sub(r'[ङञणनम]्(?=[क-ह])', 'ं', s)

def cer(expected_deva, heard_deva):
    e, h = norm_deva(expected_deva), norm_deva(heard_deva)
    return levenshtein(e, h) / max(1, len(e))

In [ ]:
for w in ['ದೃಶ್ಯತ್ವ', 'ಮಿಥ್ಯಾತ್ವ', 'ಅವ್ಯಭಿಚಾರಿ ಲಿಂಗ', 'ಪಾರಮಾರ್ಥಿಕ ಸತ್ತಾ', 'ವ್ಯಾಪ್ತಿ']:
    r = sanskrit_rules(w)
    print(f'{w:20} -> {r:20} {" ".join(kn_to_ipa(x) for x in r.split())}')

## 3. Configuration
Voice names are placeholders: the model card does not state genders. Run the audition in section 5 first, then set `VOICES` and re-run from here.

In [ ]:
VOICES = {'advaita': 'Thoma', 'dvaita': 'Aakash', 'moderator': 'Raghav'}  # English voices, edit after auditioning
KANNADA_VOICE = 'Suresh'   # variant B
SANSKRIT_VOICE = 'Aryan'   # variant Q
KOKORO_VOICES = {'advaita': ('bm_george', 'b', 0.92), 'dvaita': ('am_michael', 'a', 1.0), 'moderator': ('bm_fable', 'b', 0.95)}
APPLY_SANSKRIT_RULES = True
N_TAKES = 3
SEED = 7

DESCRIPTIONS = {
    'advaita': "{name}'s voice is calm and contemplative, with a low pitch, speaking slowly and deliberately in a very clear, close-sounding recording with no background noise.",
    'dvaita': "{name} speaks at a moderate pace with a firm, confident and expressive tone, in a very clear, close-sounding recording with no background noise.",
    'moderator': "{name} speaks formally and neutrally at a measured, slightly slow pace, in a very clear, close-sounding recording with no background noise.",
    'recitation': "{name} recites slowly in a steady, reverent voice with clear articulation, in a very clear, close-sounding recording with no background noise.",
}

# From src/data/episode1.js (`en`), dashes replaced by commas as in `speech`.
SENTENCES = [
    ('advaita', 'Our reason is ದೃಶ್ಯತ್ವ, its being an object of cognition.'),
    ('advaita', 'Yet, the successful functioning of an object within experience does not, by itself, establish its ಪಾರಮಾರ್ಥಿಕ ಸತ್ತಾ, ultimate reality.'),
    ('dvaita', 'How can ದೃಶ್ಯತ್ವ be established as an ಅವ್ಯಭಿಚಾರಿ ಲಿಂಗ, an unfailing logical indicator, of ಮಿಥ್ಯಾತ್ವ?'),
    ('moderator', 'The definition of ಮಿಥ್ಯಾತ್ವ has been examined. Yet the question of ವ್ಯಾಪ್ತಿ remains unresolved.'),
]
# Scripture quotes (`quote.sa`), Devanagari.
QUOTES = [
    ('advaita', 'सदेव सोम्येदमग्र आसीदेकमेवाद्वितीयम्।'),
    ('dvaita', 'एको बहूनां यो विदधाति कामान्।'),
]

prep = (lambda t: KN_SPAN.sub(lambda m: sanskrit_rules(m.group()), t)) if APPLY_SANSKRIT_RULES else (lambda t: t)
OUT = '/content/bakeoff'
import os; os.makedirs(OUT, exist_ok=True)

## 4. Load models

In [ ]:
import numpy as np, soundfile as sf, librosa
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer, AutoModel

parler = ParlerTTSForConditionalGeneration.from_pretrained('ai4bharat/indic-parler-tts').to(device)
prompt_tok = AutoTokenizer.from_pretrained('ai4bharat/indic-parler-tts')
desc_tok = AutoTokenizer.from_pretrained(parler.config.text_encoder._name_or_path)
SR = parler.config.sampling_rate

asr = AutoModel.from_pretrained('ai4bharat/indic-conformer-600m-multilingual', trust_remote_code=True)

def tts(text, voice, style, seed=SEED):
    torch.manual_seed(seed)
    d = desc_tok(DESCRIPTIONS[style].format(name=voice), return_tensors='pt').to(device)
    p = prompt_tok(text, return_tensors='pt').to(device)
    with torch.no_grad():
        audio = parler.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                                prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
    return audio.cpu().numpy().squeeze().astype(np.float32)

def transcribe_sa(audio, sr):
    wav = torch.from_numpy(librosa.resample(audio, orig_sr=sr, target_sr=16000)).unsqueeze(0)
    return asr(wav, 'sa', 'ctc')

def best_sanskrit_take(kn_text, voice, style):
    """Render N_TAKES seeds of a Sanskrit-only text; keep the lowest character error rate."""
    expected = kn_to_deva(kn_text)
    takes = []
    for k in range(N_TAKES):
        audio = tts(kn_text, voice, style, seed=SEED + k)
        heard = transcribe_sa(audio, SR)
        takes.append((cer(expected, heard), heard, audio))
    return min(takes, key=lambda t: t[0])

def trim(audio, sr):
    return librosa.effects.trim(audio, top_db=35)[0]

def save(name, audio, sr):
    path = f'{OUT}/{name}.wav'
    sf.write(path, audio, sr)
    return path

print('Parler sampling rate:', SR)

## 5. Voice audition
One line in every candidate voice. Pick the three debater voices and set `VOICES` in section 3.

In [ ]:
from IPython.display import Audio, display, HTML

AUDITION = prep('Our reason is ದೃಶ್ಯತ್ವ, its being an object of cognition.')
CANDIDATES = ['Thoma', 'Dinesh', 'Jatin', 'Aakash', 'Kabir', 'Tarun', 'Raghav', 'Ravi', 'Vikas', 'Suresh', 'Chetan']
for name in CANDIDATES:
    print(name)
    display(Audio(save(f'audition_{name}', tts(AUDITION, name, 'dvaita'), SR)))

## 6. Render the variants

In [ ]:
results = []  # rows: sentence index, variant, file, note

for i, (who, text) in enumerate(SENTENCES):
    mixed = prep(text)
    # A: English voice, inline Kannada
    results.append((i, 'A', save(f's{i}_A', tts(mixed, VOICES[who], who), SR), ''))
    # B: Kannada voice, inline Kannada
    results.append((i, 'B', save(f's{i}_B', tts(mixed, KANNADA_VOICE, who), SR), ''))
    # C: spans rendered separately with the same voice, then joined
    parts, notes = [], []
    gap = np.zeros(int(0.06 * SR), dtype=np.float32)
    for is_sa, piece in split_spans(mixed):
        if is_sa:
            err, heard, audio = best_sanskrit_take(piece, VOICES[who], who)
            notes.append(f'{piece} → heard {heard} (CER {err:.2f})')
        else:
            audio = tts(piece, VOICES[who], who)
        parts += [trim(audio, SR), gap]
    results.append((i, 'C', save(f's{i}_C', np.concatenate(parts[:-1]), SR), '<br>'.join(notes)))
    print('rendered sentence', i)

In [ ]:
# D: Kokoro with rule-based IPA for the Sanskrit spans
from kokoro import KPipeline

pipelines = {lang: KPipeline(lang_code=lang) for lang in {v[1] for v in KOKORO_VOICES.values()}}
vocab = set(getattr(next(iter(pipelines.values())).model, 'vocab', {}) or []) or None

for i, (who, text) in enumerate(SENTENCES):
    voice, lang, speed = KOKORO_VOICES[who]
    marked = kokoro_text(text, vocab=vocab, rules=APPLY_SANSKRIT_RULES)
    audio = np.concatenate([np.asarray(a, dtype=np.float32) for _, _, a in pipelines[lang](marked, voice=voice, speed=speed)])
    results.append((i, 'D', save(f's{i}_D', audio, 24000), marked))
print('Kokoro vocabulary loaded:', vocab is not None)

In [ ]:
# Q: scripture quotes, Sanskrit voice and the debater's own voice
quote_rows = []
for j, (who, deva) in enumerate(QUOTES):
    kn = prep(deva_to_kn(deva))
    for label, voice, style in [('Sanskrit voice', SANSKRIT_VOICE, 'recitation'), ('debater voice', VOICES[who], who)]:
        err, heard, audio = best_sanskrit_take(kn, voice, style)
        quote_rows.append((j, label, save(f'q{j}_{voice}', audio, SR), f'heard {heard} (CER {err:.2f})'))
    print('rendered quote', j)

## 7. Listen and compare

In [ ]:
import base64, html

def player(path):
    b64 = base64.b64encode(open(path, 'rb').read()).decode()
    return f'<audio controls preload="none" src="data:audio/wav;base64,{b64}"></audio>'

rows = []
for i, (who, text) in enumerate(SENTENCES):
    rows.append(f'<tr><th colspan="3" style="text-align:left;padding-top:14px">{i}. {who}: {html.escape(text)}</th></tr>')
    for si, variant, path, note in results:
        if si == i:
            rows.append(f'<tr><td><b>{variant}</b></td><td>{player(path)}</td><td style="font-size:12px">{note if variant == "C" else html.escape(note)}</td></tr>')
for j, (who, deva) in enumerate(QUOTES):
    rows.append(f'<tr><th colspan="3" style="text-align:left;padding-top:14px">Quote {j}: {deva}</th></tr>')
    for qj, label, path, note in quote_rows:
        if qj == j:
            rows.append(f'<tr><td><b>{label}</b></td><td>{player(path)}</td><td style="font-size:12px">{html.escape(note)}</td></tr>')
display(HTML('<table>' + ''.join(rows) + '</table>'))

## 8. Download
Zips every clip plus a `results.json` so the chosen variant can be brought back into the repo.

In [ ]:
import json, shutil
json.dump({'voices': VOICES, 'kannada_voice': KANNADA_VOICE, 'rules': APPLY_SANSKRIT_RULES,
           'sentences': SENTENCES, 'quotes': QUOTES,
           'clips': [dict(zip(['sentence', 'variant', 'file', 'note'], r)) for r in results],
           'quote_clips': [dict(zip(['quote', 'voice', 'file', 'note'], r)) for r in quote_rows]},
          open(f'{OUT}/results.json', 'w'), ensure_ascii=False, indent=1)
shutil.make_archive('/content/bakeoff', 'zip', OUT)
from google.colab import files
files.download('/content/bakeoff.zip')

# Full render: the whole episode for the player

Renders every sentence of the episode and writes what the player's recorded-voice engine (`src/speech/studioEngine.js`) plays: one MP3 per sentence plus `manifest.json`.

1. Run sections **1–4** above (setup, helpers, config, models). The bake-off sections 5–8 can be skipped.
2. In the repo, run `npm run export-speech` and upload `notebooks/ep001-speech.json` when asked. It holds each sentence exactly as the player splits it (`text`, the lookup key) and the text to synthesise (`tts`, with Sanskrit in Kannada script from `src/data/pronunciation.js`).
3. Run the cells below. Finished sentences are skipped, so re-running resumes. Set `USE_DRIVE = True` to survive a disconnect.
4. Unzip `ep001.zip` into `public/audio/ep001/`, replacing the Kokoro files.

Each line gets a duration gate (re-rendered if far shorter or longer than expected, which catches skipped or looping words). Sanskrit-only lines (the scripture quotes) are rendered `MAX_TAKES` times and the take with the lowest IndicConformer error is kept.

In [ ]:
SPEECH_JSON = '/content/ep001-speech.json'   # notebooks/ep001-speech.json (npm run export-speech)
RENDER_VOICES = {'advaita': 'Chetan', 'dvaita': 'Dinesh', 'moderator': 'Dinesh'}
MIXED_MODE = 'A'          # 'A': whole sentence in one pass; 'C': Sanskrit spans rendered separately and joined
QUOTE_VOICE = 'debater'   # 'debater', or a Parler voice name such as 'Aryan'
MAX_TAKES = 3             # retries for the duration gate; best-of-N for Sanskrit-only lines
RERENDER = {}             # e.g. {'12-01.mp3': 1}: new seeds for lines that sound wrong; then re-run the render cell
USE_DRIVE = False         # keep renders on Google Drive

import os, json
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RENDER_OUT = '/content/drive/MyDrive/shastrartha-ep001'
else:
    RENDER_OUT = '/content/ep001'
os.makedirs(RENDER_OUT, exist_ok=True)

if not os.path.exists(SPEECH_JSON):
    from google.colab import files
    print('Upload notebooks/ep001-speech.json')
    SPEECH_JSON = '/content/' + next(iter(files.upload()))
ITEMS = json.load(open(SPEECH_JSON, encoding='utf-8'))['items']
missing = {it['speaker'] for it in ITEMS} - set(RENDER_VOICES)
assert not missing, f'No voice set for: {missing}'
print(len(ITEMS), 'sentences,', sum(it['sanskritOnly'] for it in ITEMS), 'Sanskrit-only')

In [ ]:
import subprocess, time

CHARS_PER_SECOND = 14.5  # the player's estimate (src/speech/engine.js)

def duration_ok(audio, text):
    estimate = len(text) / CHARS_PER_SECOND
    return 0.5 * estimate <= len(audio) / SR <= 2.0 * estimate

def finish(audio):
    audio = trim(audio, SR)
    return (audio * (0.89 / (float(np.max(np.abs(audio))) or 1.0))).astype(np.float32)

def voice_for(it):
    if it['sanskritOnly'] and QUOTE_VOICE != 'debater':
        return QUOTE_VOICE, 'recitation'
    return RENDER_VOICES[it['speaker']], it['speaker']

def signature(it):
    return {'tts': it['tts'], 'voice': voice_for(it)[0], 'mode': MIXED_MODE,
            'rules': APPLY_SANSKRIT_RULES, 'bump': RERENDER.get(it['file'], 0)}

def render_mixed(text, voice, style, seed):
    if MIXED_MODE == 'A':
        return tts(text, voice, style, seed=seed), ''
    parts, notes = [], []
    gap = np.zeros(int(0.06 * SR), dtype=np.float32)
    for is_sa, piece in split_spans(text):
        if is_sa:
            err, heard, audio = best_sanskrit_take(piece, voice, style)
            notes.append(f'{piece} CER {err:.2f}')
        else:
            audio = tts(piece, voice, style, seed=seed)
        parts += [trim(audio, SR), gap]
    return np.concatenate(parts[:-1]), '; '.join(notes)

def render_item(it):
    text = prep(it['tts'])
    voice, style = voice_for(it)
    seed0 = SEED + 1000 * RERENDER.get(it['file'], 0) + 100 * it['turn'] + it['sentence']
    if it['sanskritOnly']:
        takes = []
        for k in range(MAX_TAKES):
            audio = tts(text, voice, style, seed=seed0 + k)
            takes.append((cer(kn_to_deva(text), transcribe_sa(audio, SR)), k, audio))
        err, k, audio = min(takes, key=lambda t: t[0])
        return finish(audio), {'take': k, 'cer': round(err, 3)}
    for k in range(MAX_TAKES):
        audio, note = render_mixed(text, voice, style, seed0 + k)
        ok = duration_ok(trim(audio, SR), it['text'])
        if ok:
            break
    return finish(audio), {'take': k, 'durationOk': ok, 'note': note}

progress_path = f'{RENDER_OUT}/progress.json'
progress = json.load(open(progress_path, encoding='utf-8')) if os.path.exists(progress_path) else {}
t0 = time.time()
for n, it in enumerate(ITEMS):
    want = signature(it)
    done = progress.get(it['file'])
    if done and done['signature'] == want and os.path.exists(f"{RENDER_OUT}/{it['file']}"):
        continue
    audio, info = render_item(it)
    sf.write('/tmp/sentence.wav', audio, SR)
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', '/tmp/sentence.wav', '-ac', '1',
                    '-codec:a', 'libmp3lame', '-b:a', '64k', f"{RENDER_OUT}/{it['file']}"], check=True)
    progress[it['file']] = {'signature': want, 'duration': round(len(audio) / SR, 3), **info}
    json.dump(progress, open(progress_path, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    flag = '' if info.get('durationOk', True) else '  ⚠ duration'
    print(f"{n + 1}/{len(ITEMS)} {it['file']} {progress[it['file']]['duration']:.1f}s{flag}  {it['text'][:60]}")
print(f'done in {(time.time() - t0) / 60:.1f} min')

### Review
Listen through, add any line that sounds wrong to `RERENDER` in the config cell, re-run the config and render cells, then come back here. Warnings mark lines that failed the duration gate or had a high Sanskrit error.

In [ ]:
import base64, html
from IPython.display import HTML, display

def mp3_player(path):
    return f'<audio controls preload="none" src="data:audio/mpeg;base64,{base64.b64encode(open(path, "rb").read()).decode()}"></audio>'

rows = []
for it in ITEMS:
    p = progress.get(it['file'])
    if not p:
        rows.append(f"<tr><td>{it['file']}</td><td colspan='3'>not rendered</td></tr>")
        continue
    warn = (not p.get('durationOk', True)) or p.get('cer', 0) > 0.15
    info = ', '.join(f'{k} {v}' for k, v in p.items() if k not in ('signature', 'duration') and v != '')
    rows.append(f"<tr style=\"{'background:#fde8e8' if warn else ''}\"><td>{it['file']}<br><small>{it['speaker']}</small></td>"
                f"<td>{mp3_player(f'{RENDER_OUT}/' + it['file'])}</td><td>{html.escape(it['tts'])}</td>"
                f"<td style='font-size:12px'>{html.escape(info)}</td></tr>")
display(HTML('<table>' + ''.join(rows) + '</table>'))

### Package for the repo
Writes `manifest.json` and downloads `ep001.zip`. Unzip it into `public/audio/ep001/`. Don't run `npm run render-audio` afterwards: it would re-render everything with Kokoro.

In [ ]:
import zipfile

absent = [it['file'] for it in ITEMS if it['file'] not in progress]
assert not absent, f'Not rendered yet: {absent}'
manifest = {'model': 'indic-parler-tts', 'items': [
    {'turn': it['turn'], 'sentence': it['sentence'], 'file': it['file'], 'text': it['text'],
     'voice': progress[it['file']]['signature']['voice'], 'duration': progress[it['file']]['duration']}
    for it in ITEMS]}
json.dump(manifest, open(f'{RENDER_OUT}/manifest.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
with zipfile.ZipFile('/content/ep001.zip', 'w') as z:
    for name in [it['file'] for it in ITEMS] + ['manifest.json']:
        z.write(f'{RENDER_OUT}/{name}', name)
print(f"{len(ITEMS)} clips, {sum(i['duration'] for i in manifest['items']) / 60:.1f} min of speech")
from google.colab import files
files.download('/content/ep001.zip')

## Not covered here
- **Chanted quotes (Vāgdhenu):** try the verses in the [Vāgdhenu demo](https://huggingface.co/spaces/prathoshap/vagdhenu-demo); it needs its own CUDA 12.1 / Python 3.10 stack.
- **IndicF5 voice cloning** of a debater for the quotes: a follow-up if variant Q's debater voice is weak.